#SaaS Product Analytics Pipeline
This notebook builds an end-to-end data pipeline using:

PySpark (ingestion) SQL (transformations & analytics) Medallion Architecture (Bronze → Silver → Gold) Dataset: SaaS Product Dashboard (MAU, feature usage, MRR)

## 0. Setup
Sets up the working environment by creating the following <br>
Catalog: saas <br>
Schema: bronze, silver, gold <br>

In [0]:
%sql
CREATE CATALOG saas

In [0]:
%sql
USE CATALOG saas

In [0]:
%sql 
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

## 1. Load raw Data as tables to bronze layer

In [0]:
%sql 
SHOW VOLUMES

database,volume_name


In [0]:
base_path = "/Volumes/saas/bronze/raw_data/"

files = {
    "events": "events.csv",
    "subscriptions": "subscriptions.csv"
}

for table, file in files.items():
    df = spark.read.csv(base_path + file, header=True, inferSchema=True)
    df.write.format("delta").mode("overwrite").saveAsTable(f"saas.bronze.{table}")

print("Bronze tables created!")

Bronze tables created!


In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,events,false
bronze,subscriptions,false
bronze,users,false


In [0]:
bronze_tables = [
    "events",
    "subscriptions"
]

for table in bronze_tables:
    df = spark.sql(f"SELECT * FROM adzuna.bronze.{table} LIMIT 10")
    print(f"Table: {table}")
    display(df)

Table: events


user_id,event_date,event_type
20,2022-01-01T00:00:00.000Z,feature_C
98,2022-01-01T01:00:00.000Z,feature_A
32,2022-01-01T02:00:00.000Z,feature_A
60,2022-01-01T03:00:00.000Z,feature_A
78,2022-01-01T04:00:00.000Z,feature_A
81,2022-01-01T05:00:00.000Z,login
13,2022-01-01T06:00:00.000Z,feature_A
93,2022-01-01T07:00:00.000Z,feature_A
72,2022-01-01T08:00:00.000Z,feature_C
80,2022-01-01T09:00:00.000Z,feature_A


Table: subscriptions


user_id,signup_date,country,plan,subscription_status,revenue,churn_date
1,2022-01-01,USA,free,free,0,null
2,2022-01-02,India,free,free,0,null
3,2022-01-03,USA,pro,paid,50,null
4,2022-01-04,India,pro,paid,50,null
5,2022-01-05,USA,basic,paid,20,null
6,2022-01-06,USA,basic,paid,20,2022-06-01
7,2022-01-07,USA,free,free,0,null
8,2022-01-08,Canada,basic,paid,20,null
9,2022-01-09,USA,basic,paid,20,null
10,2022-01-10,UK,basic,paid,20,null


Table: users


user_id,signup_date,country,plan
1,2022-01-01,USA,free
2,2022-01-02,India,free
3,2022-01-03,USA,pro
4,2022-01-04,India,pro
5,2022-01-05,USA,basic
6,2022-01-06,USA,basic
7,2022-01-07,USA,free
8,2022-01-08,Canada,basic
9,2022-01-09,USA,basic
10,2022-01-10,UK,basic


In [0]:
from pyspark.sql import functions as F

for table in bronze_tables:
    df = spark.sql(f"SELECT * FROM adzuna.bronze.{table}")
    null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
    print(f"Null counts for table: {table}")
    display(null_counts)

Null counts for table: events


user_id,event_date,event_type
0,0,0


Null counts for table: subscriptions


user_id,signup_date,country,plan,subscription_status,revenue,churn_date
0,0,0,0,0,0,74


Null counts for table: users


user_id,signup_date,country,plan
0,0,0,0


### Data Exploration

In [0]:
%sql
SELECT DISTINCT event_type
FROM saas.bronze.events;

event_type
feature_C
feature_A
login
feature_B


In [0]:
%sql
SELECT DISTINCT plan, subscription_status
FROM saas.bronze.subscriptions;

plan,subscription_status
free,free
pro,paid
basic,paid


## 2. Load data from Bronze Layer to Silver Layer

In [0]:
%sql
CREATE OR REPLACE TABLE saas.silver.events AS
SELECT 
  user_id,
  CAST(event_date AS DATE) AS event_date, 
  event_type
FROM saas.bronze.events;

CREATE OR REPLACE TABLE saas.silver.subscriptions AS
SELECT * FROM saas.bronze.subscriptions;


num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Sense Check
SELECT * 
FROM saas.silver.events
LIMIT 10;

user_id,event_date,event_type
20,2022-01-01,feature_C
98,2022-01-01,feature_A
32,2022-01-01,feature_A
60,2022-01-01,feature_A
78,2022-01-01,feature_A
81,2022-01-01,login
13,2022-01-01,feature_A
93,2022-01-01,feature_A
72,2022-01-01,feature_C
80,2022-01-01,feature_A


## 3. Load data from Silver Layer to Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.events AS
SELECT * FROM saas.silver.events;

CREATE OR REPLACE TABLE saas.gold.subscriptions AS
SELECT * FROM saas.silver.subscriptions;

num_affected_rows,num_inserted_rows


### Data Analysis

**1. Monthly Active Users (MAU):**
Unique users who triggered an event in a month

In [0]:
%sql
SELECT 
  DATE_FORMAT(event_date, 'yyyy-MM') AS Month,
  COUNT(DISTINCT user_id) AS MAU
FROM saas.gold.events
GROUP BY 1
ORDER BY 1

Month,MAU
2022-01,100
2022-02,91


_**Findings:** <br>_
While the current MAU analysis provides a snapshot of our immediate engagement, two months of data represents a 'baseline' rather than a 'trend.' To provide actionable insights, we require a broader longitudinal view—ideally 6+ months. This will allow us to filter out month-to-month volatility and accurately identify our Retention Rate, which is the true driver of long-term product-led growth.

**2. Feature Adoption Analysis:**
- Which feature is most used? 
- What % of users adopt each feature?

In [0]:
%sql 
SELECT 
  event_type, 
  COUNT(DISTINCT user_id) AS num_of_distinct_users, 
  ROUND(
    (COUNT(DISTINCT user_id) * 1.0 / 
    (SELECT COUNT(DISTINCT user_id) FROM saas.gold.events) * 100), 2) AS feature_adoption_rate
FROM saas.gold.events
WHERE event_type != 'login'
GROUP BY 1
ORDER BY 2, 3 DESC;

event_type,num_of_distinct_users,feature_adoption_rate
feature_A,89,89.00
feature_C,91,91.00
feature_B,93,93.00


_**Findings:**_

**Data Normalization Note:**
To ensure the analysis reflects true product utility, event_type = login has been excluded from the primary feature-usage metrics. Since authentication is a mandatory prerequisite for all platform interactions, its inclusion would artificially inflate engagement scores and obscure the relative performance of high-value tools.

**Comparative Feature Adoption:**
The data reveals a remarkably tight engagement corridor, with only a 4% variance between the most and least utilized features.

- **Primary Driver (93%): feature_B**. This is currently the most integrated component of the user workflow.

- **Secondary Driver (91%): feature_C**. Maintains high-velocity usage, closely trailing the primary feature.

- **Tertiary Driver (89%): feature_A**. While still boasting high adoption, it represents the current floor of functional engagement.

**Strategic Elaboration & Action Points** <br>
While the high adoption rates across the board suggest a "sticky" product, the narrow margin between the "Best" and "Worst" performing features requires a deeper qualitative dive.

- **Investigating the "Frictionless" Nature of feature_B** <br>
feature_B’s 93% dominance suggests it may be the "Path of Least Resistance." <br> **Action:** Determine if feature_B is high-performing because of its superior UI/UX, or if it is simply a mandatory step in a common business process. If it is a "Gateway Feature," we can leverage its high traffic to cross-promote underutilized tools.

- **Diagnosing the feature_A "Lag"** <br>
An 89% adoption rate is objectively high, but in a mature analytics environment, being the "lowest" often signals a discoverability or latency issue. <br> **Action:** Conduct a "Time-to-Value" (TTV) analysis for feature_A. If users take longer to achieve a result in feature_A compared to feature_B, the 4% gap likely represents users who find the effort-to-reward ratio unfavorable.

- **Identifying "Core" vs. "Edge" Users** <br>
With such similar percentages, we need to verify if the 7–11% of non-users are the same group of people across all features. <br> **Action:** Perform a User Segment Overlay. If a specific cohort (e.g., junior staff or external auditors) is consistently in the "non-user" bracket, it may indicate a training gap rather than a feature flaw.

**3. Feature Usage vs Retention:** Do users who use certain features retain longer?

In [0]:
%sql
SELECT 
  MAX(signup_date),
  MAX(churn_date) 
FROM saas.gold.subscriptions

MAX(signup_date),MAX(churn_date)
2022-04-10,2022-06-26


In [0]:
%sql
WITH feature_users AS (
    -- users who used the feature at least once
    SELECT DISTINCT e.user_id, e.event_type
    FROM saas.gold.events e
    JOIN saas.gold.subscriptions s ON e.user_id = s.user_id
),
feature_churned AS (
    -- users who churned
    SELECT DISTINCT user_id
    FROM saas.gold.subscriptions s
    WHERE churn_date IS NOT NULL
)
SELECT
    f.event_type,
    COUNT(DISTINCT fc.user_id) AS churned_users,
    COUNT(DISTINCT f.user_id) AS total_users,
    ROUND((COUNT(DISTINCT fc.user_id) * 1.0 / COUNT(DISTINCT f.user_id))*100,2) AS churn_rate
FROM feature_users f
LEFT JOIN feature_churned fc
    ON f.user_id = fc.user_id
WHERE f.event_type != 'login'
GROUP BY f.event_type
ORDER BY churn_rate DESC;

event_type,churned_users,total_users,churn_rate
feature_A,25,89,28.09
feature_C,25,91,27.47
feature_B,24,93,25.81


_**Findings:**_<br>
**Comparative Churn Rate:** 

Users engaging with Feature B exhibit the lowest churn rate (25.81%), suggesting it plays a key role in product retention. In contrast, Feature A users show the highest churn (28.09%), indicating potential opportunities to improve feature value or onboarding.

Together with results shown in **2.Feature Adoption Analysis**, this metrics reveal a strong positive correlation between feature adoption and retention. Feature B emerges as the top performer, achieving the highest adoption rate (93%) and the lowest churn rate (25.81%). Conversely, Feature A shows the weakest performance, with the lowest adoption (89%) and the highest churn (28.09%). Feature C maintains a mid-tier position across both metrics, confirming that higher feature utility consistently aligns with reduced customer attrition. <br>

**Potential Hypotheses:**
* **Feature Value:** Feature B may be more critical to users' workflows, increasing product stickiness.
* **Engagement Depth:** Users interacting with Feature B may be more active users overall, which correlates with lower churn.
* **User Segment Differences:** Different user types may use different features <br>
Feature A: new users experimenting <br>
Feature B: power users <br>
Feature C: occasional users <br>
Further segmentation (by plan, country) could validate this



**Strategic Elaboration & Action Points** <br>
* **Investigate Feature A's Friction:** Since Feature A has the highest churn and lowest adoption, perform a user friction audit. Is the UI confusing, or is the "time to value" too long?
* **Identify Feature B’s "Sticky" Factors:** Conduct qualitative interviews with users of Feature B. Understanding why this feature drives retention can help you apply those same mechanics to Features A and C.
* **Run Cohort Analysis:** Measure churn among users who adopt Feature B early and users who never use Feature B. This can validate whether Feature B is causally related to retention.
* **Cross-Feature Correlation:** Analyze if users who adopt both B and C have an even lower churn rate than those who use B alone. This can help inform your "onboarding path" for new users.
* **Statistical Significance Test:** Ensure the ~2.2% difference in churn between Feature B and Feature A is statistically significant and not just seasonal noise before making major pivot decisions.

**4. Churn Rate Analysis**

_**4.1 Churn Rate by Country**_

In [0]:
%sql
SELECT * 
FROM saas.gold.subscriptions 
LIMIT 10;

user_id,signup_date,country,plan,subscription_status,revenue,churn_date
1,2022-01-01,USA,free,free,0,null
2,2022-01-02,India,free,free,0,null
3,2022-01-03,USA,pro,paid,50,null
4,2022-01-04,India,pro,paid,50,null
5,2022-01-05,USA,basic,paid,20,null
6,2022-01-06,USA,basic,paid,20,2022-06-01
7,2022-01-07,USA,free,free,0,null
8,2022-01-08,Canada,basic,paid,20,null
9,2022-01-09,USA,basic,paid,20,null
10,2022-01-10,UK,basic,paid,20,null


In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_churn_rate_by_country AS
SELECT 
  country, 
  ROUND((COUNT(CASE WHEN churn_date IS NOT NULL THEN 1 END) * 1.0 /
  COUNT(*))*100, 2) AS churn_rate
FROM saas.gold.subscriptions 
GROUP BY 1;

SELECT * 
FROM saas.gold.subscriptions_churn_rate_by_country
ORDER BY 2 DESC;

country,churn_rate
India,30.43
Canada,27.59
UK,26.09
USA,20.00


**_Findings:_**
* **India has the highest churn rate (30.43%)**, suggesting potential issues such as payment friction, different pricing sensitivity, or lack of localized features.
* **USA shows the lowest churn (20%)**, indicating stronger customer retention

**_4.2 Churn Rate by Plan_**

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_churn_rate_by_plan AS
SELECT 
  plan, 
  ROUND((COUNT(CASE WHEN churn_date IS NOT NULL THEN 1 END) * 1.0 /
  COUNT(*))*100, 2) AS churn_rate
FROM saas.gold.subscriptions 
GROUP BY 1;

SELECT * 
FROM saas.gold.subscriptions_churn_rate_by_plan
ORDER BY 2 DESC;

plan,churn_rate
basic,27.03
pro,26.67
free,25.00


_**Findings**_
* Basic plan users have the highest churn rate (27.03%), followed by Pro users (26.67%), while Free users have the lowest churn rate (25%)

**Possible Explanations:** 
* Basic users may not be experiencing enough product value after upgrading from the free plan. Many users treat Basic as a “testing phase” before deciding whether to fully commit.
* Pro users likely receive higher perceived value and are more invested in the product.
* Free users technically cannot cancel a paid subscription, so their churn behavior is different. Instead, they become inactive or stop using the product. This means free churn is often underestimated unless inactivity churn is measured.

**Strategic Elaboration & Action Points:**
* Improve Basic Plan Value to shift focus on onboarding, feature discovery and early activation
* Identify Feature Usage of Churned Basic Users to identify activation gaps


**_4.3 Chun Rate by Country + Plan_**

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_churn_rate_by_country_plan AS
SELECT 
  country,
  plan, 
  ROUND((COUNT(CASE WHEN churn_date IS NOT NULL THEN 1 END) * 1.0 /
  COUNT(*))*100, 2) AS churn_rate
FROM saas.gold.subscriptions 
GROUP BY 1, 2
ORDER BY 3 DESC;

SELECT * 
FROM saas.gold.subscriptions_churn_rate_by_country_plan
ORDER BY 3 DESC;

country,plan,churn_rate
UK,pro,50.00
USA,basic,44.44
India,free,38.46
India,pro,33.33
Canada,pro,33.33
Canada,free,30.00
UK,basic,25.00
Canada,basic,23.08
UK,free,23.08
India,basic,14.29


**_Findings:_** <br>

While the Basic plan shows a higher overall churn rate than the Pro plan in 4.2 Churn Rate by Plan, segmentation by country reveals that several of the highest churn segments belong to the Pro plan. This suggests that churn patterns are strongly influenced by regional differences rather than plan type alone. <br>

This indicates that country-level factors likely play a major role, such as: 
* market fit
* pricing sensitivity
* payment issues
* customer expectations

Multiple Pro segments (UK pro, India pro and Canada pro) appear in the highest churn rankings, indicating that
* Pro plan is overpriced in these markets
* Pro users in these countries do not perceive enough value

**_4.4 Calculate user counts, churn contribution_**

_4.4.1 Calculate Churn Rate by Segment_

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_churn_rate_by_segment AS
SELECT
    country,
    plan,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN churn_date IS NOT NULL THEN user_id END) AS churned_users,
    ROUND((COUNT(DISTINCT CASE WHEN churn_date IS NOT NULL THEN user_id END) * 1.0
        / COUNT(DISTINCT user_id)) * 100, 2) AS churn_rate
FROM saas.gold.subscriptions
GROUP BY country, plan
ORDER BY churn_rate DESC;

SELECT * 
FROM saas.gold.subscriptions_churn_rate_by_segment;

country,plan,total_users,churned_users,churn_rate
UK,pro,2,1,50.00
USA,basic,9,4,44.44
India,free,13,5,38.46
Canada,pro,6,2,33.33
India,pro,3,1,33.33
Canada,free,10,3,30.00
UK,basic,8,2,25.00
UK,free,13,3,23.08
Canada,basic,13,3,23.08
India,basic,7,1,14.29


_4.4.2 Calculate Churn Contribution_

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_churn_contribution AS
SELECT
    country,
    plan,
    COUNT(DISTINCT user_id) AS churned_users,
    ROUND((COUNT(DISTINCT user_id) * 1.0
        / (SELECT COUNT(DISTINCT user_id)
    FROM saas.gold.subscriptions
    WHERE churn_date IS NOT NULL)) * 100, 2) AS churn_contribution
FROM saas.gold.subscriptions
WHERE churn_date IS NOT NULL
GROUP BY country, plan
ORDER BY churn_contribution DESC;

SELECT * 
FROM saas.gold.subscriptions_churn_contribution;

country,plan,churned_users,churn_contribution
India,free,5,19.23
USA,basic,4,15.38
UK,free,3,11.54
Canada,basic,3,11.54
Canada,free,3,11.54
Canada,pro,2,7.69
UK,basic,2,7.69
USA,free,1,3.85
UK,pro,1,3.85
India,basic,1,3.85


In [0]:
%sql
SELECT 
  scrbs.country,
  scrbs.plan, 
  scrbs.churn_rate,
  scc.churn_contribution
FROM saas.gold.subscriptions_churn_rate_by_segment scrbs
LEFT JOIN saas.gold.subscriptions_churn_contribution scc
ON scrbs.country = scc.country
AND scrbs.plan = scc.plan;


country,plan,churn_rate,churn_contribution
UK,pro,50.00,3.85
USA,basic,44.44,15.38
India,free,38.46,19.23
Canada,pro,33.33,7.69
India,pro,33.33,3.85
Canada,free,30.00,11.54
UK,basic,25.00,7.69
UK,free,23.08,11.54
Canada,basic,23.08,11.54
India,basic,14.29,3.85


_**Findings:**_ <br>
The UK Pro segment shows the highest churn rate at 50%, indicating that half of the users on this plan eventually cancel their subscriptions. However, this segment contributes only 3.85% to overall churn, suggesting that the number of users on the UK Pro plan is relatively small. As a result, although the churn rate is high, its overall impact on total churn is limited. <br>

In contrast, the USA Basic and India Free segments exhibit both high churn rates and high churn contribution. The USA Basic plan has a 44.44% churn rate and contributes 15.38% of total churn, while the India Free plan shows a 38.36% churn rate and contributes 19.23% of total churn. <br>

This indicates that churn in these two segments is both severe and widespread. Not only are users in these groups leaving at relatively high rates, but the segments themselves represent a large portion of the user base, meaning their cancellations significantly affect overall churn. <br>

From a business perspective, this suggests that USA Basic and India Free users should be prioritised for churn reduction efforts. The combination of high churn rate and high contribution implies these segments represent the largest opportunity to improve overall retention. Potential areas for further investigation include product value perception, feature adoption, onboarding effectiveness, and pricing suitability for these specific customer groups.

_4.4.3 Adding Revenue Metrics to the Segment Table_

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_revenue_churn_rate_by_segment AS
SELECT
    country,
    plan,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE 
        WHEN churn_date IS NOT NULL THEN user_id 
    END) AS churned_users,
    SUM(revenue) AS total_revenue,
    SUM(CASE 
        WHEN churn_date IS NOT NULL THEN revenue 
    END) AS churned_revenue,
    ROUND((COUNT(DISTINCT CASE 
        WHEN churn_date IS NOT NULL THEN user_id 
    END) * 1.0 / COUNT(DISTINCT user_id)) * 100, 2) AS churn_rate,
    ROUND((SUM(CASE 
        WHEN churn_date IS NOT NULL THEN revenue
    END) * 1.0 / SUM(revenue))*100, 2) AS revenue_churn_rate
FROM saas.gold.subscriptions
WHERE plan != 'free' -- excluding this condition because there's no revenue earned when plan is free
GROUP BY country, plan;

SELECT * 
FROM saas.gold.subscriptions_revenue_churn_rate_by_segment
ORDER BY revenue_churn_rate DESC;

country,plan,total_users,churned_users,total_revenue,churned_revenue,churn_rate,revenue_churn_rate
UK,pro,2,1,100,50,50.00,50.00
USA,basic,9,4,180,80,44.44,44.44
India,pro,3,1,150,50,33.33,33.33
Canada,pro,6,2,300,100,33.33,33.33
UK,basic,8,2,160,40,25.00,25.00
Canada,basic,13,3,260,60,23.08,23.08
India,basic,7,1,140,20,14.29,14.29
USA,pro,4,0,200,null,0.00,null


**_Notes:_** <br>
Revenue churn mirrors customer churn because revenue per user is fixed within each plan tier. As a result, revenue losses scale proportionally with customer churn across segments.<br>
The Free plan is excluded from this analysis as it does not generate revenue and therefore does not contribute to revenue churn.

**_Findings:_** <br>
The UK Pro segment shows the highest revenue churn rate at 50%, indicating that half of the revenue generated from this segment was lost due to churn during the analysis period. However, the absolute churned revenue is only $50, suggesting that the revenue base for this segment is relatively small. As a result, while the revenue churn rate appears severe, the overall financial impact on the business is limited. <br>

In contrast, the USA Basic segment demonstrates both a high revenue churn rate (44.44%) and a higher absolute churned revenue of $80. This indicates that not only are customers on the Basic plan in the USA churning at a high rate, but their cancellations are also resulting in meaningful revenue loss. This combination suggests that the Basic tier represents a key revenue risk area, where improving retention could materially reduce overall revenue churn.<br>

Additionally, the Canada Pro segment shows a revenue churn rate of 33.33% with $100 in churned revenue, which is the largest revenue loss among the segments analysed. While the churn rate is slightly lower than the UK Pro and USA Basic segments, the larger revenue base means that churn in this segment results in a greater absolute financial impact.<br>

Taken together, these findings suggest that retention efforts should prioritise segments where both revenue churn rate and churned revenue are significant. In particular, the USA Basic segment should be closely investigated, as churn in this tier is both frequent and financially meaningful. Losing Basic plan customers appears to contribute disproportionately to revenue loss, indicating potential issues related to pricing-value alignment, feature limitations, or customer fit within the Basic tier.<br>

Meanwhile, although the UK Pro segment has the highest revenue churn rate, its relatively small revenue base makes it a lower immediate priority compared with segments that generate larger revenue losses.<br>

Next steps for deeper analysis would include examining user engagement and feature adoption among Basic plan users, as well as assessing whether these customers are failing to reach key activation milestones before churning. Identifying behavioural differences between retained and churned users in this tier could help uncover the primary drivers of revenue loss.

_4.4.4 Calculate Revenue Contribution_

In [0]:
%sql
CREATE OR REPLACE TABLE saas.gold.subscriptions_revenue_churn_contribution AS
SELECT
    country,
    plan,
    COUNT(DISTINCT user_id) AS churned_users,
    ROUND((COUNT(DISTINCT user_id) * 1.0
        / (SELECT COUNT(DISTINCT user_id)
    FROM saas.gold.subscriptions
    WHERE churn_date IS NOT NULL)) * 100, 2) AS churn_contribution,
    SUM(CASE 
        WHEN churn_date IS NOT NULL THEN revenue 
    END) AS churned_revenue,
    ROUND(SUM(CASE 
        WHEN churn_date IS NOT NULL THEN revenue 
    END)
    /
    SUM((SELECT SUM(CASE 
        WHEN churn_date IS NOT NULL THEN revenue 
    END) FROM saas.gold.subscriptions))*100, 2)
    AS revenue_churn_contribution
FROM saas.gold.subscriptions
WHERE plan != 'free' -- excluding this condition because there's no revenue earned when plan is free
GROUP BY country, plan
ORDER BY revenue_churn_contribution DESC;

SELECT * 
FROM saas.gold.subscriptions_revenue_churn_contribution

country,plan,churned_users,churn_contribution,churned_revenue,revenue_churn_contribution
UK,pro,2,7.69,50,6.25
Canada,pro,6,23.08,100,4.17
India,pro,3,11.54,50,4.17
USA,basic,9,34.62,80,2.22
UK,basic,8,30.77,40,1.25
Canada,basic,13,50.00,60,1.15
India,basic,7,26.92,20,0.71
USA,pro,4,15.38,null,null


In [0]:
%sql
SELECT 
srcrbs.*, 
srcc.revenue_churn_contribution
FROM saas.gold.subscriptions_revenue_churn_rate_by_segment srcrbs
LEFT JOIN saas.gold.subscriptions_revenue_churn_contribution srcc
ON srcrbs.country = srcc.country 
AND srcrbs.plan = srcc.plan

country,plan,total_users,churned_users,total_revenue,churned_revenue,churn_rate,revenue_churn_rate,revenue_churn_contribution
Canada,pro,6,2,300,100,33.33,33.33,4.17
UK,basic,8,2,160,40,25.00,25.00,1.25
UK,pro,2,1,100,50,50.00,50.00,6.25
USA,basic,9,4,180,80,44.44,44.44,2.22
Canada,basic,13,3,260,60,23.08,23.08,1.15
India,basic,7,1,140,20,14.29,14.29,0.71
USA,pro,4,0,200,null,0.00,null,null
India,pro,3,1,150,50,33.33,33.33,4.17


**5. Revenue Analysis (MRR)**
Sum of subscription revenue per month
* revenue by plan 
* revenue by country 
* revenue growth

**_5.1 Revenue by plan_**

In [0]:
%sql
SELECT
    plan,
    SUM(revenue) AS mrr
FROM saas.gold.subscriptions
GROUP BY plan
ORDER BY mrr DESC;

plan,mrr
pro,750
basic,740
free,0


**_Findings:_** <br>
Analysis of revenue by plan shows that the Basic and Pro tiers generate a similar level of total revenue, with Pro contributing $750 and Basic contributing $740, while the Free plan generates no revenue. As expected, the Free tier is excluded from revenue-based analyses since it does not directly contribute to the company’s revenue. <br>

The near-equal revenue contribution from the Basic and Pro plans indicates that both tiers play a similarly important role in the company’s revenue base. This suggests that retention and growth efforts should not focus solely on higher-tier plans, as losing Basic customers can have a comparable financial impact to losing Pro customers. <br>

Given that the Basic tier contributes nearly the same total revenue as Pro, churn within the Basic segment represents a meaningful revenue risk, particularly when combined with the previously observed high revenue churn rate in certain Basic segments (e.g., USA Basic). This reinforces the need to closely monitor customer behaviour within the Basic plan, as improvements in retention within this tier could have a significant impact on overall revenue stability. <br>

Overall, the findings suggest that both Pro and Basic plans should be key focus areas for retention and revenue optimisation, as they collectively represent the primary sources of revenue for the business. Further analysis could explore differences in engagement, feature usage, and lifecycle behaviour between Basic and Pro users to better understand what drives retention and upgrade opportunities across these tiers.

**_5.2 Revenue by Country_**

In [0]:
%sql
SELECT
    country,
    SUM(revenue) AS mrr
FROM saas.gold.subscriptions
GROUP BY country
ORDER BY mrr DESC;

country,mrr
Canada,560
USA,380
India,290
UK,260


**_Findings:_**<br>
Revenue distribution by country shows that Canada generates the highest total revenue at $560, followed by USA ($380), India ($290), and UK ($260). While Canada appears to be the strongest revenue-generating market, this does not necessarily mean that retention and growth efforts should focus exclusively on Canada.<br>

Canada’s high revenue contribution suggests that the product has strong adoption and monetisation in this market, making it an important segment to maintain and protect. However, the comparatively lower revenue generated from India and the UK indicates potential opportunities for growth or areas where the product may not be performing as strongly.<br>

Lower revenue contribution from these markets could be driven by several factors, such as smaller user bases, lower plan adoption, higher churn rates, or differences in pricing sensitivity and product-market fit. Understanding whether the issue stems from acquisition, conversion to paid plans, or retention would be important for identifying the appropriate strategy.<br>

Therefore, while Canada represents a key revenue base to sustain, further investigation into India and the UK markets could reveal opportunities to increase revenue, either by improving conversion from free to paid plans, enhancing engagement among existing users, or addressing potential barriers to adoption. This approach ensures the business focuses not only on protecting its strongest market but also on unlocking revenue growth in underperforming regions.<br>

**_5.3 Revenue Growth_**

In [0]:
%sql
SELECT
    DATE_FORMAT(signup_date, 'yyyy-MM') AS signup_month,
    SUM(revenue) AS MRR
FROM saas.gold.subscriptions
GROUP BY 1
ORDER BY 1;

signup_month,MRR
2022-01,440
2022-02,520
2022-03,400
2022-04,130


**_Findings:_**<br>
Revenue trends over the four-month period show that revenue remained relatively stable during the first three months, increasing from $440 in Month 1 to $520 in Month 2, before slightly declining to $400 in Month 3. This suggests that the business maintained a consistent revenue base during this period, with only moderate fluctuations.<br>

However, Month 4 shows a significant drop in revenue to $130, representing a substantial decline compared to the previous months. This sharp decrease indicates a potential disruption in the revenue trend and warrants further investigation.<br>

Several factors could explain this decline. For example, the drop may be driven by seasonality effects, such as lower demand during certain periods of the year, or it could be related to changes in customer behaviour, including increased churn, reduced new subscriptions, or fewer upgrades to paid plans. It may also reflect operational or business factors, such as the end of promotional campaigns, pricing changes, or reduced marketing and acquisition activity.<br>

To better understand the underlying cause, further analysis should examine changes in new user sign-ups, churn rates, and plan upgrades during Month 4, as well as any external factors such as seasonal demand patterns or marketing campaigns. Identifying the drivers behind this sudden drop will be important for determining whether the decline is a temporary fluctuation or an emerging retention or acquisition issue that requires intervention.

**6. Cohort Retention Analysis:** how long users stay active after signing up

In [0]:
%sql
WITH user_cohorts AS ( -- Step 1: Assign Users to a Cohort (signup_month)
    SELECT
        user_id,
        DATE_FORMAT(signup_date, 'yyyy-MM') AS cohort_month
    FROM saas.gold.subscriptions
), 

user_activity AS ( -- Step 2: Get User Activity by Month
    SELECT DISTINCT
        user_id,
        DATE_FORMAT(event_date, 'yyyy-MM') AS activity_month
    FROM saas.gold.events
),

cohort_activity AS ( -- Step 3: Join Cohort with User Activity
    SELECT
        c.user_id,
        c.cohort_month,
        a.activity_month,
        MONTHS_BETWEEN(a.activity_month, c.cohort_month) AS month_number
    FROM user_cohorts c
    JOIN user_activity a
    ON c.user_id = a.user_id
)

SELECT
    cohort_month,
    month_number,
    COUNT(DISTINCT user_id) AS active_users
FROM cohort_activity
GROUP BY 1,2
ORDER BY 1,2;

cohort_month,month_number,active_users
2022-01,0.0,31
2022-01,1.0,29
2022-02,-1.0,28
2022-02,0.0,23
2022-03,-2.0,31
2022-03,-1.0,30
2022-04,-3.0,10
2022-04,-2.0,9
